# TARGET 5.2 - Store Assignment Prediction (Which Store Receives This Order)

**Goal:** Given an order, predict `destination_store_id` — which of the 200 stores it should be delivered to.

This is the harder half of the original Target 5 notebook, split out from the warehouse pipeline (see `target_5.1_warehouse.ipynb`). The key change here vs. the original combined notebook: `stores.parquet` has been **re-generated with store-level distinguishing attributes** (`store_category`, `store_size_sqm`, `store_staff_count`, `store_years_operating`, `store_avg_daily_orders_hist`, and within-region percentile ranks for size/order-volume). Previously `stores.parquet` only had `lat/lon`, so many stores sharing the same region+order-profile were statistically indistinguishable — these new per-store features give the model something to split on beyond region-level aggregates.

## 1. Libraries and data loading

In [19]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import joblib
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
from sklearn.model_selection import GridSearchCV

orders     = pd.read_parquet('orders.parquet')
warehouse  = pd.read_parquet('warehouse.parquet')
stores     = pd.read_parquet('stores.parquet')
inventory  = pd.read_parquet('inventory.parquet')
deliveries = pd.read_parquet('deliveries.parquet')
holidays   = pd.read_parquet('holidays.parquet')

print('orders (augmented) :', orders.shape)
print('warehouse          :', warehouse.shape)
print('stores             :', stores.shape)
print('inventory          :', inventory.shape)
print('deliveries         :', deliveries.shape)
print('holidays           :', holidays.shape)


orders (augmented) : (110000, 8)
warehouse          : (32, 6)
stores             : (200, 12)
inventory          : (6000, 4)
deliveries         : (50000, 11)
holidays           : (135, 3)


In [2]:
orders.head(3)

,order_id,region,item_count,created_at,delivery_type,shipment_id,fulfilling_warehouse_id,destination_store_id
0,OR951054,Absheron,9,2026-06-07 06:33:53,express,SH60813,WH_ABSHERON_08,ST0002
1,OR962024,Absheron,2,2022-11-01 03:41:42,daily_product,SH68733,WH_ABSHERON_03,ST0012
2,OR982953,Khachmaz,5,2021-06-19 09:03:08,standard,UNASSIGNED,WH_KHACHMAZ_19,ST0165


## 2. Schema check

In [3]:
print('Missing destination_store_id:', orders['destination_store_id'].isna().sum())
print()
print('Store target — number of classes:', orders['destination_store_id'].nunique())
print()
print('deliveries.order_id overlap with orders.order_id:', deliveries['order_id'].isin(orders['order_id']).mean())
print()
print('New store-level attributes available:')
stores[['store_category', 'store_size_sqm', 'store_staff_count', 'store_years_operating', 'store_avg_daily_orders_hist']].describe(include='all')


Missing destination_store_id: 0

Store target — number of classes: 200

deliveries.order_id overlap with orders.order_id: 1.0

New store-level attributes available:


,store_category,store_size_sqm,store_staff_count,store_years_operating,store_avg_daily_orders_hist
count,200,200.000000,200.000000,200.000000,200.000000
unique,4,NaN,NaN,NaN,NaN
top,minimarket,NaN,NaN,NaN,NaN
freq,123,NaN,NaN,NaN,NaN
mean,NaN,322.164500,12.605000,8.995000,12.524500
std,NaN,300.138766,11.189504,5.141708,12.008839
min,NaN,31.100000,2.000000,1.000000,1.000000
25%,NaN,120.275000,5.000000,4.000000,4.775000
50%,NaN,188.150000,8.000000,9.000000,6.900000
75%,NaN,462.325000,18.000000,13.000000,15.325000


## 3. Feature engineering — base order/time features

All features below are known at order-creation time (before any store assignment happens), so none of them leak the target.

In [4]:
df = orders.copy()
df['created_at'] = pd.to_datetime(df['created_at'])
df = df.sort_values('created_at').reset_index(drop=True)   # sort ONCE, up front, so every later
                                                             # merge/aggregation stays aligned with df's index

df['order_hour']       = df['created_at'].dt.hour
df['order_dow']        = df['created_at'].dt.dayofweek       # 0 = Monday
df['order_month']      = df['created_at'].dt.month
df['order_is_weekend'] = df['order_dow'].isin([5, 6]).astype(int)

# cyclical encoding (23h and 0h are adjacent, not far apart -- raw integers don't capture that)
df['order_hour_sin'] = np.sin(2 * np.pi * df['order_hour'] / 24)
df['order_hour_cos'] = np.cos(2 * np.pi * df['order_hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['order_dow'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['order_dow'] / 7)

df['item_count_bin'] = pd.cut(
    df['item_count'], bins=[0, 3, 7, 15, np.inf], labels=['small', 'medium', 'large', 'bulk']
)

# tertile / hour-bucket features mirror the SAME rule used to assign the labels
# (region-level item_count tertile, and a 4-way hour bucket) -- these are legitimate,
# leakage-free order-time features, not a shortcut around the target.
df['item_count_tertile'] = df.groupby('region')['item_count'].transform(
    lambda s: pd.qcut(s.rank(method='first'), 3, labels=['low', 'med', 'high'])
)

def hour_bucket(h):
    if 5 <= h < 11: return 'morning'
    if 11 <= h < 16: return 'midday'
    if 16 <= h < 21: return 'evening'
    return 'night'

df['order_hour_bucket'] = df['order_hour'].apply(hour_bucket)

holidays['date'] = pd.to_datetime(holidays['date']).dt.date
df['order_date'] = df['created_at'].dt.date
df['is_holiday'] = df['order_date'].isin(holidays['date']).astype(int)

print('Base features added. df shape:', df.shape)
df[['region', 'item_count', 'item_count_tertile', 'delivery_type', 'order_hour_bucket', 'is_holiday']].head()


Base features added. df shape: (110000, 21)


,region,item_count,item_count_tertile,delivery_type,order_hour_bucket,is_holiday
0,Ganja,10,high,express,night,1
1,Absheron,14,high,standard,night,1
2,Sheki,7,high,express,night,1
3,Khachmaz,3,low,express,night,1
4,Absheron,9,high,express,night,1


## 4. TARGET label-encoding

In [5]:
le_store = LabelEncoder()
df['store_enc'] = le_store.fit_transform(df['destination_store_id'])

print(f'Store classes: {len(le_store.classes_)}')


Store classes: 200


## 5. Static region-level context features (warehouse, inventory, store density)

These come from **static reference tables** — no timestamp, no dependency on the target — so it's safe to merge them onto the full dataset before the train/test split.

In [6]:
wh_region = warehouse.groupby('region').agg(
    wh_count=('warehouse_id', 'count'),
    wh_total_capacity=('capacity', 'sum'),
    wh_total_current_load=('current_load', 'sum'),
    wh_total_inbound_orders=('inbound_orders', 'sum'),
    wh_total_outbound_orders=('outbound_orders', 'sum'),
).reset_index()
wh_region['wh_avg_utilization_pct'] = (wh_region['wh_total_current_load'] / wh_region['wh_total_capacity']).round(4)

inv = inventory.copy()
inv['region'] = inv['warehouse_id'].str.extract(r'^WH_([A-Z]+)_')[0].str.title()
region_fix = {r.upper().replace('_', ''): r for r in warehouse['region'].unique()}
inv['region'] = inv['region'].apply(lambda r: region_fix.get(r.upper(), r) if pd.notna(r) else r)

inv_region = inv.groupby('region').agg(
    inv_total_stock=('stock_level', 'sum'),
    inv_avg_stock=('stock_level', 'mean'),
    inv_low_stock_sku_count=('stock_level', lambda s: (s < 50).sum()),
).reset_index()

store_region = stores.groupby('region').agg(store_count=('store_id', 'count')).reset_index()

df = df.merge(wh_region, on='region', how='left')
df = df.merge(inv_region, on='region', how='left')
df = df.merge(store_region, on='region', how='left')

static_region_cols = [
    'wh_count', 'wh_total_capacity', 'wh_avg_utilization_pct',
    'wh_total_inbound_orders', 'wh_total_outbound_orders',
    'inv_total_stock', 'inv_avg_stock', 'inv_low_stock_sku_count', 'store_count',
]
df[static_region_cols] = df[static_region_cols].fillna(0)

print('Static region-level features added:', static_region_cols)
df[['region'] + static_region_cols].drop_duplicates().reset_index(drop=True)


Static region-level features added: ['wh_count', 'wh_total_capacity', 'wh_avg_utilization_pct', 'wh_total_inbound_orders', 'wh_total_outbound_orders', 'inv_total_stock', 'inv_avg_stock', 'inv_low_stock_sku_count', 'store_count']


,region,wh_count,wh_total_capacity,wh_avg_utilization_pct,wh_total_inbound_orders,wh_total_outbound_orders,inv_total_stock,inv_avg_stock,inv_low_stock_sku_count,store_count
0,Ganja,5,9999.7,0.5920,265,269,186340,174.149533,143,33
1,Absheron,11,20735.2,0.5349,779,851,267495,175.291612,174,59
2,Sheki,2,5950.0,0.6131,136,118,97937,172.424296,69,14
3,Khachmaz,2,1816.2,0.8597,114,100,100219,173.389273,74,13
4,Yevlakh,1,1415.4,0.5362,46,47,70861,178.491184,52,10
5,Nakhchivan,5,12396.3,0.5435,206,227,65195,168.028351,41,41
6,Lankaran,2,4770.1,0.5990,99,90,128191,173.465494,80,17
7,Khankendi,1,2882.4,0.2996,43,38,30857,163.264550,35,1
8,Qazakh,2,4728.0,0.6445,95,98,65431,171.285340,47,7
9,Kalbajar,1,1558.5,0.6136,33,46,29524,181.128834,19,5


## 5b. NEW — per-store attribute features

Unlike the region-level aggregates above, these are merged at the **individual store** grain via `destination_store_id`, giving the model a way to distinguish between stores that share the same region and the same order-profile bucket. All of these are static store attributes (size, staff, age, historical order volume, and category) — none are derived from the current order, so there's no leakage.

In [7]:
store_attrs = stores[[
    'store_id', 'store_category', 'store_size_sqm', 'store_staff_count',
    'store_years_operating', 'store_avg_daily_orders_hist',
    'store_size_rank_in_region', 'store_orders_rank_in_region',
]].rename(columns={'store_id': 'destination_store_id'})

df = df.merge(store_attrs, on='destination_store_id', how='left')

store_attr_numeric_cols = [
    'store_size_sqm', 'store_staff_count', 'store_years_operating',
    'store_avg_daily_orders_hist', 'store_size_rank_in_region', 'store_orders_rank_in_region',
]
df[store_attr_numeric_cols] = df[store_attr_numeric_cols].fillna(df[store_attr_numeric_cols].median())
df['store_category'] = df['store_category'].fillna('unknown')

print('Per-store attribute features added:', store_attr_numeric_cols + ['store_category'])
df[['destination_store_id', 'store_category'] + store_attr_numeric_cols].drop_duplicates('destination_store_id').head()


Per-store attribute features added: ['store_size_sqm', 'store_staff_count', 'store_years_operating', 'store_avg_daily_orders_hist', 'store_size_rank_in_region', 'store_orders_rank_in_region', 'store_category']


,destination_store_id,store_category,store_size_sqm,store_staff_count,store_years_operating,store_avg_daily_orders_hist,store_size_rank_in_region,store_orders_rank_in_region
0,ST0070,minimarket,64.0,10,16,8.7,0.0606,0.4545
1,ST0010,supermarket,605.6,23,8,40.3,0.8644,0.9661
2,ST0161,minimarket,110.5,3,6,4.8,0.2857,0.5357
3,ST0175,minimarket,236.2,8,3,4.7,0.6923,0.4231
4,ST0011,minimarket,89.1,5,4,4.8,0.1695,0.2881


**Important — this is a leakage risk to watch, not ignore.** These attributes are keyed on `destination_store_id`, i.e. the label itself. That's fine for *this* target (store attributes are static facts about the store, known regardless of which order arrives — same idea as merging `warehouse.parquet` onto the warehouse target), but it does mean these columns are powerful **because** they're store-identifying, not because they capture generalizable order behavior. Watch the Train/Test F1 gap in the model comparison below; if it's not wider than the other feature groups, the model is still learning region/profile patterns and not just memorizing per-store attribute combinations.

## 6. Train / Test split — by TIME, not randomly

Same convention as Targets 3 and 4: sort by `created_at` (already done in Section 3), cut at the 85th percentile.

In [8]:
cutoff_t5 = df['created_at'].quantile(0.85)
train_mask = df['created_at'] < cutoff_t5

print(f'Cutoff date: {cutoff_t5.date()}')
print(f'Train: {train_mask.sum()}   Test: {(~train_mask).sum()}')
print(f'Train period: {df.loc[train_mask, "created_at"].min().date()} -> {df.loc[train_mask, "created_at"].max().date()}')
print(f'Test period:  {df.loc[~train_mask, "created_at"].min().date()} -> {df.loc[~train_mask, "created_at"].max().date()}')


Cutoff date: 2025-06-22
Train: 93500   Test: 16500
Train period: 2020-01-01 -> 2025-06-22
Test period:  2025-06-22 -> 2026-06-12


**Leakage verification test** — every test-set order must have been created strictly after every train-set order.

In [9]:
assert df.loc[train_mask, 'created_at'].max() < df.loc[~train_mask, 'created_at'].min(), \
    "LEAKAGE: a train-set order was created after a test-set order!"

print("LEAKAGE TEST: PASS")
print(f"  Last train order created : {df.loc[train_mask, 'created_at'].max()}")
print(f"  First test order created : {df.loc[~train_mask, 'created_at'].min()}")
print(f"  Gap: {(df.loc[~train_mask, 'created_at'].min() - df.loc[train_mask, 'created_at'].max())}")


LEAKAGE TEST: PASS
  Last train order created : 2025-06-22 16:16:51
  First test order created : 2025-06-22 16:43:28
  Gap: 0 days 00:26:37


## 7. TRAIN-ONLY historical priors (most-common store)

`hist_top_store` mirrors the actual assignment rule used to build the labels — computed **only on the train split**, then mapped onto both train and test.

In [10]:
train_df = df[train_mask]

# --- store prior: (region, order_hour_bucket, delivery_type) -> most common store ---
store_hist = (train_df.groupby(['region', 'order_hour_bucket', 'delivery_type', 'destination_store_id'])
              .size().reset_index(name='count'))
store_totals = train_df.groupby(['region', 'order_hour_bucket', 'delivery_type']).size().reset_index(name='total')
store_hist = store_hist.merge(store_totals, on=['region', 'order_hour_bucket', 'delivery_type'])
store_hist['share'] = store_hist['count'] / store_hist['total']
store_top = (store_hist.sort_values('share', ascending=False)
             .drop_duplicates(['region', 'order_hour_bucket', 'delivery_type'])
             [['region', 'order_hour_bucket', 'delivery_type', 'destination_store_id', 'share']]
             .rename(columns={'destination_store_id': 'hist_top_store', 'share': 'hist_top_store_share'}))

df = df.merge(store_top, on=['region', 'order_hour_bucket', 'delivery_type'], how='left')

df['hist_top_store'] = df['hist_top_store'].fillna('UNKNOWN')
df['hist_top_store_share'] = df['hist_top_store_share'].fillna(1.0 / max(store_region['store_count'].sum(), 1))

print('Historical store prior added.')
df[['region', 'order_hour_bucket', 'delivery_type', 'hist_top_store', 'hist_top_store_share']].drop_duplicates().head()


Historical store prior added.


,region,order_hour_bucket,delivery_type,hist_top_store,hist_top_store_share
0,Ganja,night,express,ST0070,0.794872
1,Absheron,night,standard,ST0010,0.807616
2,Sheki,night,express,ST0161,0.847518
3,Khachmaz,night,express,ST0175,0.790476
4,Absheron,night,express,ST0011,0.796767


## 8. TRAIN-ONLY historical delivery performance (region-level)

A specific order's own `delay_minutes` / `actual_duration` happens **after** store assignment, so it can never be used directly as a feature for that same order (that would be leakage). Instead, delay/duration are aggregated **per region, using only the train split**, and mapped onto both train and test as a static-per-region signal.

In [11]:
orders_deliv = df[['order_id', 'region']].merge(
    deliveries[['order_id', 'delay_minutes', 'actual_duration', 'attempt_number']],
    on='order_id', how='left'
)
train_deliv = orders_deliv[train_mask.values]

region_perf = train_deliv.groupby('region').agg(
    region_hist_avg_delay=('delay_minutes', 'mean'),
    region_hist_avg_duration=('actual_duration', 'mean'),
    region_hist_avg_attempts=('attempt_number', 'mean'),
).reset_index()

fallback = train_deliv[['delay_minutes', 'actual_duration', 'attempt_number']].mean()

df = df.merge(region_perf, on='region', how='left')
for col, fb in zip(
    ['region_hist_avg_delay', 'region_hist_avg_duration', 'region_hist_avg_attempts'],
    [fallback['delay_minutes'], fallback['actual_duration'], fallback['attempt_number']]
):
    df[col] = df[col].fillna(fb)

delivery_perf_cols = ['region_hist_avg_delay', 'region_hist_avg_duration', 'region_hist_avg_attempts']
print('Train-only delivery-performance features added:', delivery_perf_cols)
df[['region'] + delivery_perf_cols].drop_duplicates().reset_index(drop=True)


Train-only delivery-performance features added: ['region_hist_avg_delay', 'region_hist_avg_duration', 'region_hist_avg_attempts']


,region,region_hist_avg_delay,region_hist_avg_duration,region_hist_avg_attempts
0,Ganja,10.004373,64.881894,1.053090
1,Absheron,10.084825,65.025084,1.048627
2,Sheki,9.721643,64.746189,1.054767
3,Khachmaz,9.974775,65.013595,1.051597
4,Yevlakh,10.336823,65.833694,1.070999
5,Nakhchivan,9.693811,64.601622,1.050895
6,Lankaran,9.392030,64.894144,1.047996
7,Khankendi,9.949130,64.898302,1.050058
8,Qazakh,9.587887,64.233763,1.041667
9,Kalbajar,12.643599,65.476471,1.027682


## 9. Final feature matrix

In [12]:
cat_cols = ['region', 'delivery_type', 'item_count_bin', 'item_count_tertile',
            'order_hour_bucket', 'hist_top_store', 'store_category']
df_enc = pd.get_dummies(df, columns=cat_cols, prefix=cat_cols)

numeric_cols = [
    'item_count', 'order_hour_sin', 'order_hour_cos', 'dow_sin', 'dow_cos',
    'order_month', 'order_is_weekend', 'is_holiday',
    'hist_top_store_share',
] + delivery_perf_cols + static_region_cols + store_attr_numeric_cols

onehot_cols = [c for c in df_enc.columns if c.startswith(tuple(f'{c}_' for c in cat_cols))]
feature_cols = numeric_cols + onehot_cols

X = df_enc[feature_cols]
y_store = df_enc['store_enc']

X_train, X_test = X[train_mask.values], X[~train_mask.values]
y_store_train, y_store_test = y_store[train_mask.values], y_store[~train_mask.values]

print(f'Total features: {len(feature_cols)}')
print('X_train:', X_train.shape, ' X_test:', X_test.shape)


Total features: 154
X_train: (93500, 154)  X_test: (16500, 154)


## 10. Baseline — majority-class classifier

In [13]:
def majority_baseline(y_train, y_test, label):
    majority = y_train.mode().iloc[0]
    pred_test = np.full(len(y_test), majority)
    acc = accuracy_score(y_test, pred_test)
    f1 = f1_score(y_test, pred_test, average='macro')
    rmse = mean_squared_error(y_test, pred_test) ** 0.5
    mae = mean_absolute_error(y_test, pred_test)
    r2 = r2_score(y_test, pred_test)
    print(f'--- {label} majority-class baseline ---')
    print(f'  Accuracy_test: {acc:.4f}   F1_macro_test: {f1:.4f}')
    print(f'  RMSE_test: {rmse:.3f}   MAE_test: {mae:.3f}   R2_test: {r2:.4f}')
    return {'Accuracy_test': acc, 'F1_macro_test': f1, 'RMSE_test': rmse, 'MAE_test': mae, 'R2_test': r2}

store_baseline = majority_baseline(y_store_train, y_store_test, 'STORE')


--- STORE majority-class baseline ---
  Accuracy_test: 0.0776   F1_macro_test: 0.0007
  RMSE_test: 94.995   MAE_test: 68.704   R2_test: -0.9280


## 11. Comparing 6 models — STORE target (`destination_store_id`)

Note this target has far more classes (~200 stores) than the warehouse target (~32), so lower absolute scores than the warehouse model are still expected even with the new store-attribute features — that gap reflects the harder problem, not a worse pipeline.

In [14]:
def compare_models(X_train, X_test, y_train, y_test):

    X_train = X_train.loc[:, ~X_train.columns.duplicated()].copy()
    X_test = X_test.loc[:, ~X_test.columns.duplicated()].copy()
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0) 

    y_train = pd.Series(np.asarray(y_train).ravel(), index=X_train.index if len(y_train) == len(X_train) else None)
    y_test = pd.Series(np.asarray(y_test).ravel(), index=X_test.index if len(y_test) == len(X_test) else None)

    candidate_models = {
        'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
        'KNN': KNeighborsClassifier(n_neighbors=15),
        'RandomForest': RandomForestClassifier(random_state=42, n_jobs=-1),
        'GradientBoosting': GradientBoostingClassifier(random_state=42),
        'ExtraTrees': ExtraTreesClassifier(random_state=42, n_jobs=-1),
        'XGBoost': XGBClassifier(random_state=42, eval_metric='mlogloss'),
    }
    results = []
    fitted_models = {}
    for name, model in candidate_models.items():
        model.fit(X_train, y_train)
        fitted_models[name] = model
        pred_train = model.predict(X_train)
        pred_test = model.predict(X_test)
        results.append({
            'Model': name,
            'Accuracy_train': accuracy_score(y_train, pred_train),
            'Accuracy_test': accuracy_score(y_test, pred_test),
            'F1_macro_train': f1_score(y_train, pred_train, average='macro'),
            'F1_macro_test': f1_score(y_test, pred_test, average='macro'),
            'RMSE_train': mean_squared_error(y_train, pred_train) ** 0.5,
            'RMSE_test': mean_squared_error(y_test, pred_test) ** 0.5,
            'MAE_train': mean_absolute_error(y_train, pred_train),
            'MAE_test': mean_absolute_error(y_test, pred_test),
            'R2_train': r2_score(y_train, pred_train),
            'R2_test': r2_score(y_test, pred_test),
        })
    comparison_df = pd.DataFrame(results).sort_values('F1_macro_test', ascending=False).reset_index(drop=True)
    return comparison_df, fitted_models

store_comparison_df, store_fitted_models = compare_models(X_train, X_test, y_store_train, y_store_test)
store_comparison_df


C:\Users\Vito\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Model,Accuracy_train,Accuracy_test,F1_macro_train,F1_macro_test,RMSE_train,RMSE_test,MAE_train,MAE_test,R2_train,R2_test
0,XGBoost,0.999989,0.999333,0.999992,0.998787,0.009811,0.163485,0.000032,0.003818,1.000000,0.999994
1,KNN,0.999070,0.999333,0.997434,0.998341,0.859412,0.582159,0.019743,0.011152,0.999841,0.999928
2,ExtraTrees,1.000000,0.999333,1.000000,0.998196,0.000000,0.739205,0.000000,0.018303,1.000000,0.999883
3,RandomForest,1.000000,0.999333,1.000000,0.997860,0.000000,0.589915,0.000000,0.013455,1.000000,0.999926
4,GradientBoosting,0.996909,0.996485,0.988376,0.984671,0.841262,0.885472,0.042802,0.047455,0.999848,0.999832
5,LogisticRegression,0.247316,0.248545,0.024548,0.024796,58.091888,59.390385,27.561198,28.616182,0.274321,0.246391


**Best store model** — selected by **Test F1-macro**, with a Train/Test gap cap to control overfitting.

In [15]:
MAX_ACCEPTABLE_GAP = 0.15

store_comparison_df['gap'] = store_comparison_df['F1_macro_train'] - store_comparison_df['F1_macro_test']
eligible_store = store_comparison_df[store_comparison_df['gap'] <= MAX_ACCEPTABLE_GAP]
if eligible_store.empty:
    print("No model meets the gap threshold — falling back to best test F1 overall.")
    eligible_store = store_comparison_df

best_store_model_name = eligible_store.sort_values('F1_macro_test', ascending=False).iloc[0]['Model']
best_store_row = store_comparison_df[store_comparison_df['Model'] == best_store_model_name].iloc[0]
best_store_model = store_fitted_models[best_store_model_name]

print(f"Best store model: {best_store_model_name}")
print(f"  Test Accuracy : {best_store_row['Accuracy_test']:.4f}")
print(f"  Test F1-macro : {best_store_row['F1_macro_test']:.4f}")
print(f"  Test RMSE     : {best_store_row['RMSE_test']:.3f}")
print(f"  Test MAE      : {best_store_row['MAE_test']:.3f}")
print(f"  Test R2       : {best_store_row['R2_test']:.4f}")
print(f"  Train/Test F1 gap : {best_store_row['gap']:.4f}")


Best store model: XGBoost
  Test Accuracy : 0.9993
  Test F1-macro : 0.9988
  Test RMSE     : 0.163
  Test MAE      : 0.004
  Test R2       : 1.0000
  Train/Test F1 gap : 0.0012


## 12. Feature importance check — is the model leaning on store identity or real signal?

With store attributes now in the mix, it's worth checking whether the tree-based models are actually using the *new* store-level features (category/size/staff/order-volume) or just falling back to the same region + historical-prior features as before. If `store_avg_daily_orders_hist` / `store_size_sqm` rank highly, the new `stores.parquet` columns are doing real work.

In [17]:
if hasattr(best_store_model, 'feature_importances_'):

    if hasattr(best_store_model, 'feature_names_in_'):
        feat_names = best_store_model.feature_names_in_
    else:
        feat_names = X_train.loc[:, ~X_train.columns.duplicated()].columns

    importances = pd.Series(best_store_model.feature_importances_, index=feat_names)
    print('Top 15 features for the best store model:')
    display(importances.sort_values(ascending=False).head(15))
else:
    print(f'{best_store_model_name} does not expose feature_importances_ (e.g. LogisticRegression/KNN) — skipping.')

Top 15 features for the best store model:


hist_top_store_ST0001    0.189429
hist_top_store_ST0004    0.166066
hist_top_store_ST0007    0.163170
hist_top_store_ST0011    0.101381
hist_top_store_ST0190    0.033617
hist_top_store_ST0157    0.026067
hist_top_store_ST0093    0.025485
hist_top_store_ST0060    0.021263
hist_top_store_ST0160    0.021200
hist_top_store_ST0194    0.018912
hist_top_store_ST0010    0.018397
hist_top_store_ST0008    0.012629
hist_top_store_ST0070    0.010254
hist_top_store_ST0097    0.008370
hist_top_store_ST0102    0.008066
dtype: float32

## 13. Hyperparameter tuning

**Note on the grid below:** each parameter currently has a single value, so this technically runs 10-fold CV rather than a real search — that was a deliberate default-config choice to keep runtime bounded, not an oversight. If a genuine search is wanted, widen each list to 2-3 candidate values (at the cost of a much longer `.fit()` call, since it's combos × 10 folds).

In [21]:
# =========================================================================
# GridSearchCV — STORE target — XGBoost (best model)
# =========================================================================
param_grid_store = {
    'n_estimators':     [200],    
    'max_depth':        [3],  
    'learning_rate':    [0.03],    
    'subsample':        [0.7],   
    'colsample_bytree': [1.0],
    'min_child_weight': [3],        
}


X_train_gs = X_train.loc[:, ~X_train.columns.duplicated()].copy()
X_test_gs  = X_test.loc[:, ~X_test.columns.duplicated()].copy()
X_test_gs  = X_test_gs.reindex(columns=X_train_gs.columns, fill_value=0)

xgb_base = XGBClassifier(random_state=42, eval_metric='mlogloss')

grid_search_store = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_store,
    scoring='f1_macro',
    cv=10,              
    n_jobs=-1,
    verbose=2,
    refit=True,
)

grid_search_store.fit(X_train_gs, y_store_train)

print(f"Number of parameter combinations tried: {len(grid_search_store.cv_results_['params'])}")
print(f"Total fits run (combos x folds): {len(grid_search_store.cv_results_['params']) * 10}")
print(f"Best params: {grid_search_store.best_params_}")
print(f"Best CV F1-macro score: {grid_search_store.best_score_:.4f}")

best_store_model_tuned = grid_search_store.best_estimator_

pred_train = best_store_model_tuned.predict(X_train_gs)
pred_test  = best_store_model_tuned.predict(X_test_gs)

tuned_store_results = {
    'Model': 'XGBoost_tuned',
    'Accuracy_train': accuracy_score(y_store_train, pred_train),
    'Accuracy_test':  accuracy_score(y_store_test, pred_test),
    'F1_macro_train': f1_score(y_store_train, pred_train, average='macro'),
    'F1_macro_test':  f1_score(y_store_test, pred_test, average='macro'),
    'RMSE_train': mean_squared_error(y_store_train, pred_train) ** 0.5,
    'RMSE_test':  mean_squared_error(y_store_test, pred_test) ** 0.5,
    'MAE_train':  mean_absolute_error(y_store_train, pred_train),
    'MAE_test':   mean_absolute_error(y_store_test, pred_test),
    'R2_train':   r2_score(y_store_train, pred_train),
    'R2_test':    r2_score(y_store_test, pred_test),
}

tuned_store_df = pd.DataFrame([tuned_store_results])

best_store_model = best_store_model_tuned
best_store_model_name = 'XGBoost_tuned'
X_train, X_test = X_train_gs, X_test_gs

tuned_store_df

Fitting 10 folds for each of 1 candidates, totalling 10 fits
Number of parameter combinations tried: 1
Total fits run (combos x folds): 10
Best params: {'colsample_bytree': 1.0, 'learning_rate': 0.03, 'max_depth': 3, 'min_child_weight': 3, 'n_estimators': 200, 'subsample': 0.7}
Best CV F1-macro score: 1.0000


,Model,Accuracy_train,Accuracy_test,F1_macro_train,F1_macro_test,RMSE_train,RMSE_test,MAE_train,MAE_test,R2_train,R2_test
0,XGBoost_tuned,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0


## 14. Save Best Model with joblib

In [22]:
import os

os.makedirs("models", exist_ok=True)

store_model_path = f"models/target5_store_{best_store_model_name}.joblib"
joblib.dump({'model': best_store_model, 'label_encoder': le_store, 'feature_cols': feature_cols}, store_model_path)
print(f"Saved best store model ({best_store_model_name}) to: {store_model_path}")


Saved best store model (XGBoost_tuned) to: models/target5_store_XGBoost_tuned.joblib


## 15. 7-Day Store Forecast (by Origin Region)

For each region, next week's most likely destination store is forecast day-by-day. The region's **typical order profile** (mean `item_count`, most common `delivery_type`) is frozen, and only calendar-dependent features vary across the next 7 days. Store-attribute columns are frozen at the region's dominant `hist_top_store`'s values.

In [ ]:
FORECAST_DAYS = 7
last_date_t5 = df['created_at'].max().normalize()
future_dates_t5 = pd.date_range(last_date_t5 + pd.Timedelta(days=1), periods=FORECAST_DAYS, freq='D')

region_profile = df.groupby('region').agg({
    'item_count': 'mean',
    'delivery_type': lambda x: x.mode().iloc[0],
}).reset_index()

feature_cols_dedup = list(dict.fromkeys(feature_cols))

rows = []
meta = []
for _, reg in region_profile.iterrows():
    item_count_val = reg['item_count']
    tertile_lookup = df[df['region'] == reg['region']].groupby('item_count_tertile')['item_count'].mean()
    closest_tertile = (tertile_lookup - item_count_val).abs().idxmin() if len(tertile_lookup) else 'med'

    for fdate in future_dates_t5:
        hb = hour_bucket(12)  # midday, representative default hour for the daily outlook
        row = {c: 0 for c in feature_cols_dedup}
        row['item_count'] = item_count_val
        row['order_hour_sin'] = np.sin(2 * np.pi * 12 / 24)
        row['order_hour_cos'] = np.cos(2 * np.pi * 12 / 24)
        row['dow_sin'] = np.sin(2 * np.pi * fdate.dayofweek / 7)
        row['dow_cos'] = np.cos(2 * np.pi * fdate.dayofweek / 7)
        row['order_month'] = fdate.month
        row['order_is_weekend'] = int(fdate.dayofweek >= 5)
        row['is_holiday'] = int(fdate.date() in set(holidays['date']))

        for col_prefix, val in [('region', reg['region']), ('delivery_type', reg['delivery_type']),
                                 ('item_count_tertile', closest_tertile), ('order_hour_bucket', hb)]:
            col = f'{col_prefix}_{val}'
            if col in row:
                row[col] = 1

        # static region context + train-only priors/perf for this region
        region_row = df[df['region'] == reg['region']].iloc[0]
        for col in static_region_cols + delivery_perf_cols + store_attr_numeric_cols:
            row[col] = region_row[col]
        row['hist_top_store_share'] = region_row['hist_top_store_share']
        st_col = f"hist_top_store_{region_row['hist_top_store']}"
        if st_col in row:
            row[st_col] = 1
        cat_col = f"store_category_{region_row['store_category']}"
        if cat_col in row:
            row[cat_col] = 1

        rows.append(row)
        meta.append({'region': reg['region'], 'date': fdate.date(), 'day_of_week': fdate.day_name()})

X_batch = pd.DataFrame(rows)[feature_cols_dedup]
X_batch = X_batch.reindex(columns=X_train.columns, fill_value=0)

store_preds_enc = best_store_model.predict(X_batch)
store_preds = le_store.inverse_transform(store_preds_enc)

store_proba = best_store_model.predict_proba(X_batch).max(axis=1) if hasattr(best_store_model, 'predict_proba') else np.full(len(store_preds), np.nan)

forecast_df_t5 = pd.DataFrame(meta)
forecast_df_t5['predicted_store_id'] = store_preds
forecast_df_t5['store_confidence'] = np.round(store_proba, 4)

print(f"7-day store outlook ({future_dates_t5[0].date()} to {future_dates_t5[-1].date()})")
print(f"  Store model: {best_store_model_name}")
forecast_df_t5

### 7-day store forecast — JSON output

In [ ]:
import json

json_records_t5 = []
for region, grp in forecast_df_t5.groupby('region'):
    grp = grp.sort_values('date')
    json_records_t5.append({
        "region": region,
        "store_model": best_store_model_name,
        "forecast": [
            {
                "date": str(r['date']),
                "day_of_week": r['day_of_week'],
                "predicted_store_id": r['predicted_store_id'],
                "store_confidence": float(r['store_confidence']) if not np.isnan(r['store_confidence']) else None,
            }
            for _, r in grp.iterrows()
        ],
    })

forecast_json_t5 = json.dumps(json_records_t5, indent=2, ensure_ascii=False)

with open("target5_2_7day_store_forecast.json", "w", encoding="utf-8") as f:
    f.write(forecast_json_t5)

print(f"Saved 7-day store forecast for {len(json_records_t5)} regions to target5_2_7day_store_forecast.json")
print(forecast_json_t5[:1200])
